In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 독립변수 데이터 분포 점검
- 데이터의 형태가 심각하게 비정규성을 띄고
- 샘플 모델의 accuracy가 너무 낮다면
- 스케일링 통해 분포를 맞출 필요
- 또한 feature importance가 낮은데 불균등 분포여도 feature 제거 고려

# 타겟변수 분포 점검

# 데이터 전처리 for DNN

In [20]:
import pandas as pd
from pathlib import Path

# ===== 경로 설정 =====
input_path  = r"C:/Users/Admin/OneDrive/문서/카카오톡 받은 파일/WA_Fn-UseC_-HR-Employee-Attrition.csv"
output_path = r"C:/Users/Admin/OneDrive/문서/카카오톡 받은 파일/WA_Fn-UseC_-HR-Employee-Attrition_변환.csv"

# ===== 데이터 불러오기 =====
df = pd.read_csv(input_path)

# ===== 매핑 정의 =====
travel_map = {
    'Travel_Rarely': '1~29회',
    'Travel_Frequently': '30회 이상',
    'Non-Travel': '0회'
}

major_map = {
    'Life Sciences': '사회과학계열',
    'Other': '기타',
    'Medical': '자연과학계열',
    'Marketing': '상경계열',
    'Technical Degree': '공학계열',
    'Human Resources': '인문학'
}

jobrole_map = {
    'Sales Executive': '영업직',
    'Research Scientist': '연구직',
    'Laboratory Technician': '엔지니어',
    'Manufacturing Director': '제조 책임자',
    'Healthcare Representative': '연구직 관리자',
    'Manager': '인사관리자',
    'Sales Representative': '영업 담당자',
    'Research Director': '연구 관리자',
    'Human Resources': '인사사원'
}

dept_map = {
    'Sales': '영업',
    'Research & Development': 'R&D',
    'Human Resources': '사무직'
}

# 숫자/문자 모두 대비
eval_map = {3: '보통', 4: '좋다', '3': '보통', '4': '좋다'}

# ===== 유틸: 후보 열 중 존재하는 첫 번째 열 찾기 =====
def pick_col(candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# ===== 1) 출장 =====
src = pick_col(['출장', 'BusinessTravel'])
if src is not None:
    df['출장'] = df[src].replace(travel_map)

# ===== 2) 전공 =====
src = pick_col(['전공', 'EducationField'])
if src is not None:
    df['전공'] = df[src].replace(major_map)

# ===== 3) 직급 =====
src = pick_col(['직급', 'JobRole'])
if src is not None:
    df['직급'] = df[src].replace(jobrole_map)

# ===== 4) 부서 =====
src = pick_col(['부서', 'Department'])
if src is not None:
    df['부서'] = df[src].replace(dept_map)

# ===== 5) 업무평가 =====
# 보통 원본이 숫자 3/4 이지만, 문자열일 수도 있어 둘 다 매핑
src = pick_col(['업무평가', 'PerformanceRating'])
if src is not None:
    df['업무평가'] = df[src].replace(eval_map)

# ===== 저장 =====
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False, encoding='utf-8-sig')

# ===== 확인용 출력 (존재하는 컬럼만 보여주기) =====
preview_cols = [c for c in ['출장', '전공', '직급', '부서', '업무평가'] if c in df.columns]
print("✅ 변환된 CSV 저장 완료:", output_path)
if preview_cols:
    print("\n[변환 결과 미리보기]")
    print(df[preview_cols].head().to_string(index=False))


✅ 변환된 CSV 저장 완료: C:/Users/Admin/OneDrive/문서/카카오톡 받은 파일/WA_Fn-UseC_-HR-Employee-Attrition_변환.csv

[변환 결과 미리보기]
    출장     전공   직급  부서 업무평가
 1~29회 사회과학계열  영업직  영업   보통
30회 이상 사회과학계열  연구직 R&D   좋다
 1~29회     기타 엔지니어 R&D   보통
30회 이상 사회과학계열  연구직 R&D   보통
 1~29회 자연과학계열 엔지니어 R&D   보통


# 이진분류 dnn

In [22]:
import numpy as np
import pandas as pd
import random, os
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import precision_score, classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


CSV_PATH = r"C:\Users\Admin\OneDrive\바탕 화면\2차 플젝\WA_Fn-UseC_-HR-Employee-Attrition_변환.csv" 
TARGET   = "업무평가"

df = pd.read_csv(CSV_PATH, encoding="utf-8")
assert TARGET in df.columns, f"'{TARGET}' 컬럼이 없습니다. 실제 컬럼: {df.columns.tolist()}"
print(df.shape, df[TARGET].value_counts(dropna=False).sort_index())


(1470, 29) 보통    1244
좋다     226
Name: 업무평가, dtype: int64


In [23]:
y_raw = df[TARGET].copy()
X_raw = df.drop(columns=[TARGET]).copy()

# 숫자/범주형 분리
cat_cols = X_raw.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_raw.select_dtypes(include=[np.number, "bool"]).columns.tolist()

# 결측 간단 처리
for c in num_cols: X_raw[c] = X_raw[c].fillna(X_raw[c].median())
for c in cat_cols: X_raw[c] = X_raw[c].fillna("missing")

# 원-핫 인코딩
X_ohe = pd.get_dummies(X_raw, columns=cat_cols, drop_first=False)

# 이진/다중 판별 + 라벨 인코딩
unique_labels = sorted(pd.Series(y_raw).dropna().unique().tolist())
n_classes = len(unique_labels)
task = "binary" if n_classes == 2 else "multiclass"

if task == "binary":
    bin_map = {unique_labels[0]: 0, unique_labels[1]: 1}
    inv_bin_map = {v:k for k,v in bin_map.items()}
    y = y_raw.map(bin_map).astype(int)
else:
    le = LabelEncoder()
    y = le.fit_transform(y_raw)

# 분할
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_ohe, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=SEED, stratify=y_train_full
)

# 스케일링(수치형만)
scaler = StandardScaler()
if len(num_cols) > 0:
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_val[num_cols]   = scaler.transform(X_val[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

# OHE 칼럼 정합성 보정
all_cols = X_train.columns
X_val  = X_val.reindex(columns=all_cols,  fill_value=0)
X_test = X_test.reindex(columns=all_cols, fill_value=0)

input_dim = X_train.shape[1]
print(task, n_classes, input_dim)


binary 2 50


In [24]:
# class_weight (SMOTE 없이 1차)
classes = np.unique(y_train)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(k): float(v) for k,v in zip(classes, cw)}

def build_dnn(input_dim, task, num_classes=2, hidden=[512,256,128], dropout=0.3, lr=5e-4):
    model = keras.Sequential([layers.Input(shape=(input_dim,))])
    for h in hidden:
        model.add(layers.Dense(h, activation="relu"))
        model.add(layers.Dropout(dropout))
    if task == "binary":
        model.add(layers.Dense(1, activation="sigmoid"))
        loss = "binary_crossentropy"
    else:
        model.add(layers.Dense(num_classes, activation="softmax"))
        loss = "sparse_categorical_crossentropy"

    model.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=["accuracy"])
    return model

model = build_dnn(input_dim, task, num_classes=n_classes, hidden=[512,256,128], dropout=0.3, lr=5e-4)
callbacks = [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True)]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=256,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=0
)
print("훈련완료")


훈련완료


In [25]:
def evaluate_with_precision(model, X, y_true, task, threshold=0.5):
    if task == "binary":
        proba = model.predict(X, verbose=0).ravel()
        y_pred = (proba >= threshold).astype(int)
    else:
        proba = model.predict(X, verbose=0)
        y_pred = np.argmax(proba, axis=1)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    acc  = accuracy_score(y_true, y_pred)
    return prec, acc, y_pred

best_thr = 0.5
if task == "binary":
    # 정밀도 극대화용 간단 그리드
    grid = np.linspace(0.1, 0.9, 33)
    best_prec = -1.0
    for t in grid:
        p, a, _ = evaluate_with_precision(model, X_val, y_val, task, threshold=t)
        if p > best_prec:
            best_prec = p; best_thr = t

val_prec, val_acc, _ = evaluate_with_precision(model, X_val, y_val, task, threshold=best_thr)
test_prec, test_acc, y_pred_test = evaluate_with_precision(model, X_test, y_test, task, threshold=best_thr)

print("=== Validation ===")
print(f"- best threshold: {best_thr:.3f}" if task=="binary" else "- multiclass: threshold N/A")
print(f"- precision_weighted: {val_prec:.4f}")
print(f"- accuracy          : {val_acc:.4f}")

print("\n=== Test ===")
print(f"- precision_weighted: {test_prec:.4f}")
print(f"- accuracy          : {test_acc:.4f}")

print("\n[Classification Report - Test]")
print(classification_report(y_test, y_pred_test, digits=4, zero_division=0))
print("[Confusion Matrix - Test]")
print(confusion_matrix(y_test, y_pred_test))


=== Validation ===
- best threshold: 0.450
- precision_weighted: 0.7570
- accuracy          : 0.3912

=== Test ===
- precision_weighted: 0.7545
- accuracy          : 0.3673

[Classification Report - Test]
              precision    recall  f1-score   support

           0     0.8621    0.3012    0.4464       249
           1     0.1594    0.7333    0.2619        45

    accuracy                         0.3673       294
   macro avg     0.5107    0.5173    0.3542       294
weighted avg     0.7545    0.3673    0.4182       294

[Confusion Matrix - Test]
[[ 75 174]
 [ 12  33]]


# 2번째 이진분류 dnn

In [49]:
# ===========================
# 1) 라이브러리/데이터 로드
# ===========================
import numpy as np, pandas as pd, random, os
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ★ 여기만 본인 경로로 변경
CSV_PATH = r"C:\Users\Admin\OneDrive\바탕 화면\2차 플젝\WA_Fn-UseC_-HR-Employee-Attrition_변환.csv"
TARGET   = "업무평가"

df = pd.read_csv(CSV_PATH, encoding="utf-8")
assert TARGET in df.columns, f"'{TARGET}' 컬럼이 없습니다. 실제 컬럼: {df.columns.tolist()}"

print("데이터:", df.shape)
print("타깃 분포:\n", df[TARGET].value_counts(dropna=False).sort_index())


데이터: (1470, 29)
타깃 분포:
 보통    1244
좋다     226
Name: 업무평가, dtype: int64


In [50]:
# ===========================
# 2) 전처리 & 분할 (이진 고정)
# ===========================
y_raw = df[TARGET].copy()
X_raw = df.drop(columns=[TARGET]).copy()

# 숫자/범주 분리 + 결측 처리
cat_cols = X_raw.select_dtypes(include=["object","category"]).columns.tolist()
num_cols = X_raw.select_dtypes(include=[np.number, "bool"]).columns.tolist()

for c in num_cols: X_raw[c] = X_raw[c].fillna(X_raw[c].median())
for c in cat_cols: X_raw[c] = X_raw[c].fillna("missing")

# 원-핫 인코딩
X_ohe = pd.get_dummies(X_raw, columns=cat_cols, drop_first=False)

# 이진 라벨 매핑 (예: {3,4} -> {0,1})
unique_labels = sorted(pd.Series(y_raw).dropna().unique().tolist())
assert len(unique_labels)==2, f"현재 타깃 유니크 {unique_labels} → 이진분류 아님"
bin_map    = {unique_labels[0]:0, unique_labels[1]:1}
inv_bin_map= {v:k for k,v in bin_map.items()}
y = y_raw.map(bin_map).astype(int)

# 60/20/20 분할 (stratify)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_ohe, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=SEED, stratify=y_train_full
)

# 수치형만 스케일링
scaler = StandardScaler()
if len(num_cols)>0:
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_val[num_cols]   = scaler.transform(X_val[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

# OHE 컬럼 정합성
all_cols = X_train.columns
X_val  = X_val.reindex(columns=all_cols,  fill_value=0)
X_test = X_test.reindex(columns=all_cols, fill_value=0)

input_dim = X_train.shape[1]
print("input_dim:", input_dim)
print("라벨 비율:", dict(pd.Series(y).value_counts(normalize=True).round(3)))


input_dim: 50
라벨 비율: {0: 0.846, 1: 0.154}


In [51]:
# ===========================
# 3) 모델 빌더 + summary
# ===========================
def build_dnn(input_dim, hidden=[512,256,128], dropout=0.3, lr=5e-4):
    m = keras.Sequential([layers.Input(shape=(input_dim,))])
    for h in hidden:
        m.add(layers.Dense(h, activation="relu"))
        m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1, activation="sigmoid"))  # 이진분류
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
              loss="binary_crossentropy",
              metrics=["accuracy"])
    return m

tmp_model = build_dnn(input_dim)
tmp_model.summary()


Model: "sequential_90"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_360 (Dense)           (None, 512)               26112     
                                                                 
 dropout_270 (Dropout)       (None, 512)               0         
                                                                 
 dense_361 (Dense)           (None, 256)               131328    
                                                                 
 dropout_271 (Dropout)       (None, 256)               0         
                                                                 
 dense_362 (Dense)           (None, 128)               32896     
                                                                 
 dropout_272 (Dropout)       (None, 128)               0         
                                                                 
 dense_363 (Dense)           (None, 1)               

In [52]:
# ===========================
# 4) 임계값 최적화(벡터화, 예측 1번)
# ===========================
from sklearn.metrics import precision_score, accuracy_score

def best_threshold_weighted_precision_fast(proba: np.ndarray, y_true: np.ndarray):
    """
    predict()로 얻은 proba(양성 확률)과 y_true(0/1)에서
    weighted precision을 최대로 하는 임계값과 값을 O(n log n)으로 계산.
    """
    proba = proba.ravel().astype(float)
    y = y_true.astype(int)

    idx = np.argsort(-proba)           # 확률 내림차순
    y_sorted = y[idx]
    proba_sorted = proba[idx]

    n = len(y_sorted)
    total_pos = y.sum()
    total_neg = n - total_pos
    w1 = total_pos / n
    w0 = total_neg / n

    k = np.arange(1, n + 1)
    cum_TP = np.cumsum(y_sorted)       # 상위 k개 중 TP
    TP = cum_TP
    FP = k - TP
    FN = total_pos - TP
    TN = total_neg - FP

    prec1 = np.divide(TP, TP + FP, out=np.ones_like(TP, dtype=float), where=(TP + FP) > 0)
    prec0 = np.divide(TN, TN + FN, out=np.ones_like(TN, dtype=float), where=(TN + FN) > 0)

    weighted = w0 * prec0 + w1 * prec1
    best_idx = int(np.argmax(weighted))
    best_prec = float(weighted[best_idx])
    best_thr = float(proba_sorted[best_idx])  # 이 값 이상이면 1로 예측
    return best_thr, best_prec

def eval_precision_from_proba(proba, y_true, thr):
    y_pred = (proba >= thr).astype(int)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    acc  = accuracy_score(y_true, y_pred)
    return prec, acc, y_pred


In [53]:
# ===========================
# 5) 빠른 탐색(작은 그리드) → 최적 조합만 재학습
# ===========================
from itertools import product

TARGET_PRECISION = 0.85      # 목표 weighted precision
MAX_EPOCHS_COARSE = 40       # 빠른 탐색 epoch
PATIENCE_COARSE   = 5

# 1) Coarse Search: 작은 조합으로 빠르게 탐색
hidden_set  = [[512,256,128]]
dropout_set = [0.2, 0.3]
lr_set      = [5e-4, 3e-4]
batch_set   = [512]
w0_mul_set  = [1.0, 2.0, 3.0]   # 음성(0) 가중 배수 ↑ → FP 패널티 ↑ → precision 우호

# class_weight 기본
classes     = np.unique(y_train)
base_cw_arr = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
base_cw     = dict(zip(classes, base_cw_arr))

best = {"val_prec":-1, "cfg":None, "thr":0.5, "weights":None}

combos = list(product(hidden_set, dropout_set, lr_set, batch_set, w0_mul_set))
print(f"[Coarse] 총 {len(combos)} 조합")

for i, (hidden, dr, lr, bs, w0mul) in enumerate(combos, 1):
    print(f" -> ({i}/{len(combos)}) hidden={hidden}, dropout={dr}, lr={lr}, batch={bs}, w0mul={w0mul}", flush=True)
    model = build_dnn(input_dim, hidden=hidden, dropout=dr, lr=lr)

    cw = {0: base_cw.get(0,1.0)*w0mul, 1: base_cw.get(1,1.0)}
    cb = [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=PATIENCE_COARSE, restore_best_weights=True),
          keras.callbacks.TerminateOnNaN()]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS_COARSE,
        batch_size=bs,
        class_weight=cw,
        callbacks=cb,
        verbose=0
    )

    # 예측 1번만
    proba_val = model.predict(X_val, batch_size=2048, verbose=0).ravel()
    thr, val_p = best_threshold_weighted_precision_fast(proba_val, y_val.values)
    print(f"    val_precision={val_p:.4f} @thr={thr:.3f}")

    if val_p > best["val_prec"]:
        best.update(val_prec=val_p, cfg=(hidden,dr,lr,bs,w0mul), thr=thr, weights=model.get_weights())

    if val_p >= TARGET_PRECISION:
        print("✅ 목표 달성! Coarse 탐색 중단")
        break

    K.clear_session()

print("\n[Best Coarse] val_precision:", round(best["val_prec"],4), "cfg:", best["cfg"], "thr:", round(best["thr"],3))


[Coarse] 총 12 조합
 -> (1/12) hidden=[512, 256, 128], dropout=0.2, lr=0.0005, batch=512, w0mul=1.0
    val_precision=0.8727 @thr=0.431
✅ 목표 달성! Coarse 탐색 중단

[Best Coarse] val_precision: 0.8727 cfg: ([512, 256, 128], 0.2, 0.0005, 512, 1.0) thr: 0.431


In [54]:
# ===========================
# 6) 최적 조합으로 Fine 재학습 + 평가
# ===========================
(hidden, dr, lr, bs, w0mul) = best["cfg"]
model = build_dnn(input_dim, hidden=hidden, dropout=dr, lr=lr)
if best["weights"] is not None:
    model.set_weights(best["weights"])  # coarse 가중치로 시작(선택)

cw = {0: base_cw.get(0,1.0)*w0mul, 1: base_cw.get(1,1.0)}
cb = [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
      keras.callbacks.TerminateOnNaN()]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=120,
    batch_size=bs,
    class_weight=cw,
    callbacks=cb,
    verbose=0
)

# Validation 최적 임계값(벡터화)
proba_val  = model.predict(X_val,  batch_size=2048, verbose=0).ravel()
thr, val_p = best_threshold_weighted_precision_fast(proba_val, y_val.values)

# Test 평가(예측 1번)
proba_test = model.predict(X_test, batch_size=2048, verbose=0).ravel()
val_p,  val_a,  _      = eval_precision_from_proba(proba_val,  y_val.values,  thr)
test_p, test_a, y_pred = eval_precision_from_proba(proba_test, y_test.values, thr)

print("=== Validation ===")
print(f"precision_weighted: {val_p:.4f} | accuracy: {val_a:.4f} | thr: {thr:.3f}")

print("=== Test ===")
print(f"precision_weighted: {test_p:.4f} | accuracy: {test_a:.4f}")

print("\n[Classification Report - Test]")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))
print("[Confusion Matrix - Test]")
print(confusion_matrix(y_test, y_pred))


=== Validation ===
precision_weighted: 0.8719 | accuracy: 0.2143 | thr: 0.407
=== Test ===
precision_weighted: 0.6578 | accuracy: 0.1803

[Classification Report - Test]
              precision    recall  f1-score   support

           0     0.7500    0.0482    0.0906       249
           1     0.1475    0.9111    0.2539        45

    accuracy                         0.1803       294
   macro avg     0.4487    0.4797    0.1722       294
weighted avg     0.6578    0.1803    0.1156       294

[Confusion Matrix - Test]
[[ 12 237]
 [  4  41]]


In [55]:
# ===========================
# 7) 단일 샘플 추론 함수 (실사용)
# ===========================
base_cols = all_cols  # 학습 시점의 OHE 컬럼

def predict_one(sample: dict, model, thr: float):
    """
    sample 예시:
    {"나이": 35, "참여프로젝트": 12, "전공": "공학계열", ...}
    """
    row = pd.DataFrame([sample])

    # 동일 전처리
    if len(num_cols)>0:
        for c in num_cols:
            if c in row.columns:
                row[c] = row[c].fillna(df[c].median() if c in df else row[c].median())
    for c in cat_cols:
        if c in row.columns:
            row[c] = row[c].fillna("missing")
    row = pd.get_dummies(row, columns=[c for c in row.columns if c in cat_cols], drop_first=False)

    # 누락 OHE 보정 + 순서 맞추기
    for col in base_cols:
        if col not in row.columns:
            row[col] = 0
    row = row[base_cols]

    # 스케일
    if len(num_cols)>0:
        row[num_cols] = scaler.transform(row[num_cols])

    # 예측
    proba = float(model.predict(row, verbose=0).ravel()[0])
    pred01 = int(proba >= thr)
    return {"pred": inv_bin_map[pred01], "proba": proba, "thr": thr}
